# Add Phase: Token Introspection and Revocation (PingOne)

This notebook demonstrates the workflow for adding a new phase to perform token introspection from Authorize (mock and real) to PingOne, handling revoked/expired tokens with a modal, and providing a UI button to revoke the token.

In [ ]:
# 1. Import Required Libraries
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
import json

In [ ]:
# 2. Mock Token Introspection Logic
def mock_token_introspection(token):
    # Simulate token status for demo: 'active', 'revoked', 'expired'
    if token == "revoked-token":
        return {"active": False, "reason": "revoked"}
    elif token == "expired-token":
        return {"active": False, "reason": "expired"}
    else:
        return {"active": True}

In [ ]:
# 3. Real Token Introspection with PingOne
def pingone_token_introspection(token, introspect_url, client_id, client_secret):
    """
    Calls PingOne's introspection endpoint.
    Returns dict with 'active' and optional 'reason'.
    """
    try:
        resp = requests.post(
            introspect_url,
            data={"token": token},
            auth=(client_id, client_secret),
            timeout=5
        )
        resp.raise_for_status()
        data = resp.json()
        if not data.get("active", False):
            return {"active": False, "reason": data.get("reason", "revoked/expired")}
        return {"active": True}
    except Exception as e:
        return {"active": False, "reason": f"error: {e}"}

In [ ]:
# 4. UI: Modal for Revoked/Expired Token
def show_kill_switch_modal(reason):
    modal = widgets.Output()
    with modal:
        clear_output()
        display(widgets.HTML(f"""
        <div style='background:#fee2e2;padding:24px;border-radius:10px;border:2px solid #dc2626;'>
            <h2 style='color:#dc2626;'>You have hit the Kill Switch</h2>
            <p style='color:#991b1b;'>Reason: <b>{reason}</b></p>
        </div>
        """))
    display(modal)
    return modal

In [ ]:
# 5. UI: Red Button to Revoke Token at PingOne
def revoke_token_ui(token, revoke_url, client_id, client_secret):
    btn = widgets.Button(description="Revoke Token", button_style="danger")
    output = widgets.Output()

    def on_click(_):
        with output:
            clear_output()
            try:
                resp = requests.post(
                    revoke_url,
                    data={"token": token},
                    auth=(client_id, client_secret),
                    timeout=5
                )
                resp.raise_for_status()
                print("Token revoked at PingOne.")
            except Exception as e:
                print(f"Error revoking token: {e}")
    btn.on_click(on_click)
    display(btn, output)
    return btn, output

In [ ]:
# 6. Integration: Token Revocation and Modal Display
def demo_token_flow(token, introspect_url, revoke_url, client_id, client_secret, use_mock=True):
    # Step 1: Introspect token
    if use_mock:
        result = mock_token_introspection(token)
    else:
        result = pingone_token_introspection(token, introspect_url, client_id, client_secret)
    if not result["active"]:
        show_kill_switch_modal(result.get("reason", "revoked/expired"))
    else:
        print("Token is active.")
        # Show revoke button
        revoke_token_ui(token, revoke_url, client_id, client_secret)

# Example usage (mock):
demo_token_flow(
    token="revoked-token",  # Try 'revoked-token', 'expired-token', or any other string
    introspect_url="https://example.pingone.com/introspect",
    revoke_url="https://example.pingone.com/revoke",
    client_id="demo-client-id",
    client_secret="demo-client-secret",
    use_mock=True
)